# Aufgabe 3 · Belegte Verbindungen mit Neo4j

[Aufgabenstellung](README.md) · [Walkthrough](WALKTHROUGH.md) · [Aktenleseführer](../../docs/AKTENLESEFUEHRER.md)

Die Redaktion beginnt mit der Analyseakte **DOW-UAP-D077**. Welche weiteren Unterlagen erreicht sie über die im Portal gespeicherten Verweise? Und was ist mit einer solchen Verbindung tatsächlich belegt?

**Teil A und B bilden die Aufgabe. Teil C ist eine Vertiefung.**
Ergänze die markierten Cypher-Stellen und Deine fachliche Interpretation.

## 0. Eigenständiger Einstieg

Kernel: **Python (rothstein-storage-workshop-2026)**.

### Vor der ersten Codezelle: den gewählten Neo4j-Dienst verwenden

**Wähle genau einen Betriebsweg. Eine neue Desktop-Instanz ist nur für Weg B nötig, wenn noch keine passende Instanz vorhanden ist.**

| Weg | Vor dem Notebook | Graphansicht nach dem Import |
| --- | --- | --- |
| **A: lokal mit Docker** | Neo4j-Desktop-Instanzen stoppen; den eingerichteten Neo4j-Container verwenden. | [Neo4j Browser](http://localhost:7474/browser/) mit `bolt://127.0.0.1:7687` |
| **B: lokal ohne Docker** | Einen laufenden Neo4j-Kurscontainer zuerst stoppen; die Desktop-Kursinstanz starten. | Bei derselben Instanz **Open → Neo4j Browser** (1.x) bzw. **Connect → Query** (2.x) |
| **C: Codespaces** | Den automatischen Dienststart abwarten; den Host `neo4j` aus der Containerkonfiguration verwenden. | Über die weitergeleitete Codespaces-Adresse; keine lokale Desktop-Instanz starten. |

**Portkonflikt vermeiden:** Weg A und B belegen lokal standardmässig `7474` (Weboberfläche) und `7687` (Bolt). Betreibe dort nur einen Neo4j-Server. Beim Wechsel zu Weg B im bisherigen Repo-Ordner `docker compose stop neo4j` ausführen, solange noch die bisherige Compose-Konfiguration vorliegt. Andere Neo4j-Container gegebenenfalls in Docker Desktop stoppen. Docker darf für MongoDB weiterlaufen.

Die [Neo4j-Startanleitung für A, B und C](../../docs/setup/NEO4J_START.md) beschreibt Start, Verbindung, Graphansicht und den Wechsel zwischen den Wegen. **Nur für Weg B:** [Desktop einrichten](../../docs/setup/NEO4J_DESKTOP.md). Eine bestehende Kursinstanz weiterverwenden. Anzeigename: **rothstein-storage-workshop-2026**, Benutzer: `neo4j`, Standardpasswort: `Storage-Neo4j-2026`, Datenbank: `neo4j`.

Lokal verwenden die Notebooks bei den Standardwerten `127.0.0.1:7687`. Abweichende Zugangsdaten in einer vorhandenen `.env` anpassen; danach den Kernel neu starten. Codespaces setzt die internen Hosts und Ports automatisch. **Erst nach erfolgreichem Dienststart die Python-Codezellen ausführen.**

Die Dateien unter `data/input/graph/` enthalten den gemeinsamen Bestand. Ergebnisse aus SQLite oder MongoDB werden nicht benötigt. Original-PDFs und Quelldaten werden nicht verändert.


In [4]:
from pathlib import Path
import sys
import json
from datetime import datetime, timedelta, timezone
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pprint import pprint

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/neo4j_workshop.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Öffne das Notebook innerhalb des Repositories.")
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from neo4j_workshop import (prepare_graph, graph_scope, import_snapshot, graph_counts,
                           query_graph, write_pairings, draw_paths)
from storage_runtime import connection_summary
pd.set_option("display.max_colwidth", 90)
print("Neo4j-Ziel:", connection_summary()["Neo4j"])

DATASET = "uap"
IS_SOLUTION = False
SCOPE = graph_scope(DATASET,solution=IS_SOLUTION)
graph = prepare_graph(DATASET)
entries = {n["node_id"]:n for n in graph["nodes"] if n["label"]=="CatalogEntry"}
start = next(n for n in entries.values() if n["properties"]["source_key"]=="DOW-UAP-D077")
START_ID = start["node_id"]
pairings = [e for e in graph["edges"] if e["type"]=="PORTAL_PAIRS_WITH"]
completed = set()
print("Arbeitsbereich:",SCOPE)
print("Katalogeinträge:",len(entries),"Vorbereitete Portalverweise:",len(pairings))
pprint(next(e for e in pairings if e["source"]==START_ID))

Neo4j-Ziel: 127.0.0.1:7690
Arbeitsbereich: uap_task
Katalogeinträge: 375 Vorbereitete Portalverweise: 336
{'edge_id': 'pair_210a1d4cb4972837cab0',
 'properties': {'pairing_id': 'pair_210a1d4cb4972837cab0',
                'raw_reference': 'Western US Event',
                'resolution_rule': 'exact_normalized_label',
                'source_entry_id': 'entry_1138b8e2e13e953eab60',
                'source_field': 'PDF Pairing',
                'target_entry_id': 'entry_09ff9a4916893c28f38d'},
 'source': 'entry_1138b8e2e13e953eab60',
 'target': 'entry_09ff9a4916893c28f38d',
 'type': 'PORTAL_PAIRS_WITH'}


## A1 · Entitäten und vorbereiteter Grundimport

| Knotenlabel | Bedeutung |
| --- | --- |
| `StorageCatalogEntry` | Ein Katalogeintrag, identifiziert durch `node_id` / `entry_id` |
| `StorageAgency` | Die im Katalog angegebene Stelle |
| `StorageAsset` | Ein eindeutiger Dateiverweis |
| `StorageRelease` | Eine Veröffentlichungstranche |

`CATALOG_AGENCY`, `LINKS_ASSET` und `IN_RELEASE` sind vorbereitet. Die gerichteten `PORTAL_PAIRS_WITH`-Verweise ergänzt Ihr in A2.

Ein Titel oder Aktenkürzel ist kein sicherer Primärschlüssel: **FBI-UAP-D014 kommt zweimal vor**. Deshalb nutzen wir technische IDs. Eine Datei, ein Katalogeintrag und ein tatsächliches Ereignis sind unterschiedliche Einheiten.

In [5]:
first = import_snapshot(DATASET,SCOPE,include_pairings=False)
second = import_snapshot(DATASET,SCOPE,include_pairings=False)
assert first==second
assert first["nodes"]==764 and first["edges"]==1147
assert first["labels"]=={"Agency":10,"Asset":374,"CatalogEntry":375,"Release":5}
completed.add("Grundimport")
pprint(first)
print("GRUNDIMPORT OK")

{'edges': 1147,
 'labels': {'Agency': 10, 'Asset': 374, 'CatalogEntry': 375, 'Release': 5},
 'nodes': 764,
 'types': {'CATALOG_AGENCY': 375, 'IN_RELEASE': 375, 'LINKS_ASSET': 397}}
GRUNDIMPORT OK


Der Grundimport ersetzt nur den gewählten Aufgaben- beziehungsweise Lösungsbereich. Wenn Du diese Zelle später erneut ausführst, werden auch dort ergänzte Portalverweise entfernt: Führe danach A2 wieder aus. Die andere Bearbeitung und die Microblogging-Demo bleiben erhalten.

### Nach dem Import: Graph anzeigen (optional)

Sobald die Zelle oben **GRUNDIMPORT OK** meldet, sind die Daten geladen. Das Notebook hat direkt in die laufende Datenbank geschrieben; Du musst keine Datei übertragen.

1. Die Oberfläche passend zu Deinem Weg öffnen: **A:** [Neo4j Browser](http://localhost:7474/browser/) der laufenden Docker-Datenbank; **B:** bei der laufenden Desktop-Kursinstanz **Open → Neo4j Browser** (1.x) oder **Connect → Query** (2.x); **C:** [Neo4j Browser im Codespace](../../docs/setup/NEO4J_START.md#weg-c-codespaces). Immer dieselbe Instanz wie im Notebook und die Datenbank **neo4j** verwenden. Benutzer **neo4j**, Passwort **`Storage-Neo4j-2026`** beziehungsweise Dein tatsächliches Passwort.
2. Die folgende Abfrage in das **Cypher-Eingabefeld dieser Oberfläche** kopieren, mit dem Play-Schalter ausführen und im Ergebnis **Graph** wählen. Diese Abfrage wird in Neo4j ausgeführt, nicht in einer Python-Codezelle.

```cypher
MATCH p = (:StorageCatalogEntry {scope: 'uap_task', source_key: 'DOW-UAP-D077'})
          -->(:StorageNode {scope: 'uap_task'})
RETURN p LIMIT 25;
```

Du siehst zunächst die vorhandenen Verbindungen ab D077. Nach Abschluss von **A2** dieselbe Abfrage erneut ausführen: Jetzt erscheinen auch die ergänzten Portalverweise. Klicke auf eine Beziehung, um ihre Belegproperties anzusehen. Nach Änderungen im Notebook genügt es, die Abfrage erneut auszuführen.

**Für die Graphansicht keinen zweiten Neo4j-Server starten.** Notebook und Oberfläche greifen auf dieselbe Instanz zu. Daten einer lokalen Instanz und Daten im Codespace sind getrennte Bestände.

## A2 · Beziehungstyp und Richtung ergänzen

Eine Zeile aus `edges.jsonl` besitzt `source`, `target`, `edge_id` und Belegproperties. **Quelle → Ziel** bedeutet: Das Metadatenfeld des Quelleintrags nennt den Zielverweis. Es bedeutet keine zeitliche Reihenfolge und keine Kausalität.

Ersetze in der folgenden Vorlage `__TYP__` durch den vorhandenen Beziehungstyp und `__PFEIL__` durch den gerichteten Pfeil. Die Eigenschaften enthalten das originale Feld, den Rohverweis und die Auflösungsregel. Bewahre sie auf.

In [6]:
pairing_query = """
UNWIND $rows AS row
MATCH (a:StorageNode:StorageCatalogEntry {scope:$scope, node_id:row.source})
MATCH (b:StorageNode:StorageCatalogEntry {scope:$scope, node_id:row.target})
MERGE (a)-[r:__TYP__ {scope:$scope, edge_id:row.edge_id}]__PFEIL__(b)
SET r += row.properties
RETURN count(r) AS processed
"""

In [7]:
if "__TYP__" in pairing_query or "__PFEIL__" in pairing_query:
    print("OFFEN: Beziehungstyp und Richtung in A2 ergänzen.")
else:
    a = write_pairings(SCOPE,pairing_query)
    b = write_pairings(SCOPE,pairing_query)
    assert a==b==336
    assert graph_counts(SCOPE)["edges"]==1483
    completed.add("Portalverweise")
    print("336 gerichtete Portalverweise; erneutes MERGE erzeugt keine Duplikate.")

OFFEN: Beziehungstyp und Richtung in A2 ergänzen.


### Eure Begründung

Was unterscheidet `PORTAL_PAIRS_WITH` von einer Aussage wie «beschreibt dasselbe Ereignis»? Warum speichert eine fachlich passende Kante zusätzlich Belegproperties?

*Notiert hier Eure Erklärung.*

## B1 · Eine Verweiskette über zwei Schritte

Vervollständige das Muster: Folge ab D077 **genau zwei gerichteten `PORTAL_PAIRS_WITH`-Beziehungen**. Schliesse Rückwege zum Start aus. Gib die geordneten Knoten-IDs, Aktenkürzel und Kanten-IDs aus.

`*2` bezeichnet genau zwei Beziehungsschritte. `*1..2` würde zusätzlich direkte Nachbarn aufnehmen. Die Ergebniszeilen zählen Pfade; ihre Anzahl muss allgemein nicht der Anzahl unterschiedlicher Zielakten entsprechen.

In [8]:
query_two_hop = """
MATCH p = (start:StorageCatalogEntry {scope:$scope, node_id:$start_id})
          -[:__PFADMUSTER__]->(target:StorageCatalogEntry {scope:$scope})
WHERE target <> start
  AND all(n IN nodes(p) WHERE n.scope = $scope)
  AND all(r IN relationships(p) WHERE r.scope = $scope)
RETURN [n IN nodes(p) | n.node_id] AS node_ids,
       [n IN nodes(p) | n.source_key] AS source_keys,
       [r IN relationships(p) | r.edge_id] AS edge_ids
ORDER BY target.source_key, target.node_id, edge_ids
"""

In [9]:
paths = []
if "Portalverweise" not in completed or "__PFADMUSTER__" in query_two_hop:
    print("OFFEN: A2 abschliessen und Pfadmuster in B1 ergänzen.")
else:
    paths = query_graph(SCOPE,query_two_hop,start_id=START_ID)
    display(pd.DataFrame([{"Start":p["source_keys"][0],"Zwischenknoten":p["source_keys"][1],
                           "Ziel":p["source_keys"][2]} for p in paths]))
    assert len(paths)==17 and len({p["node_ids"][-1] for p in paths})==17
    assert all(p["source_keys"][1]=="Western US Event" for p in paths)
    completed.add("Pfadabfrage")
    print("17 Pfade zu 17 unterschiedlichen Zielkatalogeinträgen.")

OFFEN: A2 abschliessen und Pfadmuster in B1 ergänzen.


## B2 · Pfadauswahl darstellen und Belege prüfen

Wir zeichnen eine überschaubare Auswahl der tatsächlich gefundenen Pfade. Die vollständigen Ergebnisse stehen in der Tabelle. Alle Pfeile der Abbildung bedeuten `PORTAL_PAIRS_WITH`.

Lies für den Pfad zu D080 die Belegtabelle: Das erste Feld gehört zu D077, das zweite zum Katalogeintrag «Western US Event». Dieser Zwischenknoten ist **ein Katalogeintrag**, trotz seines ereignisartig klingenden Titels.

In [10]:
query_evidence = """
UNWIND $edge_ids AS edge_id
MATCH (a:StorageCatalogEntry {scope:$scope})-[r:PORTAL_PAIRS_WITH {scope:$scope,edge_id:edge_id}]->(b:StorageCatalogEntry {scope:$scope})
RETURN a.source_key AS source_key, a.source_row AS source_row,
       r.source_field AS source_field, r.raw_reference AS raw_reference,
       r.resolution_rule AS resolution_rule, b.source_key AS target_key,
       r.edge_id AS edge_id
ORDER BY source_key, target_key, edge_id
"""

In [11]:
if paths:
    selected_keys = {"DOW-UAP-D079","DOW-UAP-D080","FBI-UAP-D015","FBI-UAP-D016"}
    selected = [p["source_keys"] for p in paths if p["source_keys"][-1] in selected_keys]
    draw_paths(selected,"Portalverweise ab D077 · Auswahl von vier Pfaden")
    focus = next(p for p in paths if p["source_keys"][-1]=="DOW-UAP-D080")
    evidence = query_graph(SCOPE,query_evidence,edge_ids=focus["edge_ids"])
    display(pd.DataFrame(evidence))
    assert len(evidence)==2
    assert all(r["source_field"] and r["raw_reference"] and r["resolution_rule"] for r in evidence)
    completed.add("Belegprüfung")
else:
    print("OFFEN: Zuerst B1 vervollständigen.")

OFFEN: Zuerst B1 vervollständigen.


### Was kann die Redaktion daraus schliessen?

Öffne [D077](../../data/raw/originals/DOW-UAP-D077.pdf) und [D080](../../data/raw/originals/DOW-UAP-D080.pdf). Nutze den [Aktenleseführer](../../docs/AKTENLESEFUEHRER.md), insbesondere den Hinweis zu D080, Seite 5.

Formuliert eine Aussage über den **belegten Rechercheweg** und eine Aussage, die der Graph **nicht belegt**. Warum macht die gemeinsame Erreichbarkeit eine Illustration noch nicht zu einer Fotografie des beschriebenen Ereignisses?

*Notiert hier Eure beiden Aussagen und den Seitenbeleg.*

## C · Vertiefung: Pfade sind keine Zielanzahlen

Wiederhole dieselbe Abfrage ab **D080**. Zähle Ergebniszeilen und unterschiedliche Ziel-IDs getrennt. Weshalb kann eine Zielakte über mehrere Zwischenknoten erreichbar sein?

Beschränke auch diese Suche auf zwei Schritte. Eine ungerichtete Abfrage hätte eine andere Bedeutung: Sie würde Verweise unabhängig von ihrer gespeicherten Richtung verfolgen.

In [12]:
RUN_EXTENSION = False

In [13]:
if RUN_EXTENSION and "Pfadabfrage" in completed:
    alternate_start = next(n for n in entries.values() if n["properties"]["source_key"]=="DOW-UAP-D080")
    alternate = query_graph(SCOPE,query_two_hop,start_id=alternate_start["node_id"])
    distinct_targets = {p["node_ids"][-1] for p in alternate}
    print("Pfade:",len(alternate),"Unterschiedliche Ziele:",len(distinct_targets))
    assert len(alternate)==24 and len(distinct_targets)==18
    display(pd.DataFrame([{"Zwischenknoten":p["source_keys"][1],"Ziel":p["source_keys"][2]} for p in alternate]))
else:
    print("VERTIEFUNG: Noch nicht ausgeführt.")

VERTIEFUNG: Noch nicht ausgeführt.


## Abschluss

Die technischen Checks prüfen Importe und Referenzwerte. Die fachliche Interpretation besprecht Ihr im Walkthrough. Haltet Eure Modellentscheidung auch in der gemeinsamen Vergleichstabelle fest.

In [14]:
required = {"Grundimport","Portalverweise","Pfadabfrage","Belegprüfung"}
missing = sorted(required-completed)
if missing:
    print("AUFGABE OFFEN:",", ".join(missing))
else:
    print("NEO4J AUFGABE TECHNISCH OK. Pfadbedeutung und Quelleninterpretation gemeinsam besprechen.")

AUFGABE OFFEN: Belegprüfung, Pfadabfrage, Portalverweise
